In [ ]:
"""
simons_nb.ipynb

figures for simons meeting

Author: Stellina X. Ao
Created: 2026-09-21
Last Modified: 2026-09-21
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids

encoders = {k: [] for k in ["full", "mb", "mf"]}

for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(f">{sess_id}")
    e = Encoder(subj_id, sess_id, norm=True)
    e_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
    e_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

    try:
        e_mb.fit_encoder()
        e_mf.fit_encoder()

        e_mb.get_r2()
        e_mf.get_r2()

        e.fit_encoder()
        e.get_r2()
    except ValueError:
        print("not enough trials...")
        continue

    encoders["full"].append(e)
    encoders["mb"].append(e_mb)
    encoders["mf"].append(e_mf)

In [ ]:
encoders = {k: np.array(v) for k, v in encoders.items()}

# population

## r2

### single session

In [ ]:
# non-parametric significance test between mb/mf explained variance, bonferroni for region
from scipy.stats import wilcoxon, false_discovery_control

alpha = 0.05

ps = [
    wilcoxon(
        x=encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]],
        y=encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]],
    ).pvalue
    for reg in encoder.regions
]
ps_corr = false_discovery_control(ps)
ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

print("sig diff r2 between mb and mf?")
for reg in encoder.regions:
    print(reg, ps_corr[reg] < alpha, f"({ps_corr[reg]:.3f})")

In [ ]:
# distro figs
from core.viz import plot_kdes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

fig_full, ax_full = plot_kdes(
    data={
        reg: encoder.scores["encoder"][encoder.reg_idxs[reg]] for reg in encoder.regions
    },
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-0.2, 1.0],
    label=r"$r^2$, full",
)
ax_full.axvline(x=0, color="#333333", linestyle="--")

fig_strat, ax_strat = plot_kdes(
    data={
        reg: encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
        - encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]
        for reg in encoder.regions
    },
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-1.0, 1.0],
    label=r"$r^2$, (mb-mf)",
)
ax_strat.axvline(x=0, color="#333333", linestyle="--")
ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

for fext in ["svg", "png"]:
    save_fig(
        fig_full,
        FIGURES_DIR / "r2" / "distros" / subj_id / sess_id,
        fname=f"r2_distro_full-{subj_id}_{sess_id}.{fext}",
    )
    save_fig(
        fig_strat,
        FIGURES_DIR / "r2" / "distros" / subj_id / sess_id,
        fname=f"r2_distro_strat-{subj_id}_{sess_id}.{fext}",
    )

### all sessions

In [ ]:
# distro figs
fig_full, ax_full = plot_kdes(
    data={
        reg: [e.scores["encoder"][e.reg_idxs[reg]] for e in encoders["full"]]
        for reg in encoder.regions
    },
    do_sem=True,
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-0.2, 1.0],
    label=r"$r^2$, full",
)
ax_full.axvline(x=0, color="#333333", linestyle="--")

fig_strat, ax_strat = plot_kdes(
    data={
        reg: [
            e_mb.scores["encoder"][e_mb.reg_idxs[reg]]
            - e_mf.scores["encoder"][e_mf.reg_idxs[reg]]
            for (e_mb, e_mf) in zip(encoders["mb"], encoders["mf"])
        ]
        for reg in encoder.regions
    },
    do_sem=True,
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-1.0, 1.0],
    label=r"$r^2$, (mb-mf)",
)
ax_strat.axvline(x=0, color="#333333", linestyle="--")
ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

for fext in ["svg", "png"]:
    save_fig(
        fig_full,
        FIGURES_DIR / "r2" / "distros" / subj_id,
        fname=f"r2_distro_full-{subj_id}_sessavg.{fext}",
    )
    save_fig(
        fig_strat,
        FIGURES_DIR / "r2" / "distros" / subj_id,
        fname=f"r2_distro_strat-{subj_id}_sessavg.{fext}",
    )

## beta weight

### single session

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ],
            y=encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ],
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: encoder.encoder_weights[encoder.reg_idxs[reg], encoder.tv_idxs[regr]]
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ]
            - encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ]
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "bweight" / "distros" / subj_id / sess_id,
            fname=f"{regr}-bweight_distro_full-{subj_id}_{sess_id}.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "bweight" / "distros" / subj_id / sess_id,
            fname=f"{regr}-bweight_distro_strat-{subj_id}_{sess_id}.{fext}",
        )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ],
            y=encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ],
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff bweight between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

### across sessions

In [ ]:
for regr in encoder.tv_keys:
    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: [
                e.encoder_weights[e.reg_idxs[reg], e.tv_idxs[regr]]
                for e in encoders["full"]
            ]
            for reg in encoder.regions
        },
        do_sem=True,
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: [
                e_mb.encoder_weights[e_mb.reg_idxs[reg], e_mb.tv_idxs[regr]]
                - e_mf.encoder_weights[e_mf.reg_idxs[reg], e_mf.tv_idxs[regr]]
                for (e_mb, e_mf) in zip(encoders["mb"], encoders["mf"])
            ]
            for reg in encoder.regions
        },
        do_sem=True,
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "bweight" / "distros" / subj_id,
            fname=f"{regr}-bweight_distro_full-{subj_id}_sessavg.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "bweight" / "distros" / subj_id,
            fname=f"{regr}-bweight_distro_strat-{subj_id}_sessavg.{fext}",
        )

## cvr2, dr2
be aware that interaction terms (but not trials from block switch) are included

### single session

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=Encoder,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
se.get_cvr2_all()
se.get_dr2_all()

In [ ]:
se_mb = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=StrategyEncoder,
    strategy_filter="mb",
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)

se_mf = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=StrategyEncoder,
    strategy_filter="mf",
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
print("mb, cvr2")
se_mb.get_cvr2_all()
print("mb, dr2")
se_mb.get_dr2_all()

print("mf, cvr2")
se_mf.get_cvr2_all()
print("mf, dr2")
se_mf.get_dr2_all()

### all sessions

In [ ]:
ses = {k: [] for k in ["full", "mb", "mf"]}

for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(f">{sess_id}")

    try:
        se_mb_ = ShuffledEncoder(
            subj_id,
            sess_id,
            enc_class=StrategyEncoder,
            strategy_filter="mb",
            tv_keys=[
                "response",
                "rewarded",
                "response_prev",
                "rewarded_prev",
            ],
            add_interaction=True,
        )

        se_mf_ = ShuffledEncoder(
            subj_id,
            sess_id,
            enc_class=StrategyEncoder,
            strategy_filter="mf",
            tv_keys=[
                "response",
                "rewarded",
                "response_prev",
                "rewarded_prev",
            ],
            add_interaction=True,
        )

        se_ = ShuffledEncoder(
            subj_id,
            sess_id,
            enc_class=Encoder,
            tv_keys=[
                "response",
                "rewarded",
                "response_prev",
                "rewarded_prev",
            ],
            add_interaction=True,
        )

        print(">> mb, cvr2")
        se_mb_.get_cvr2_all(n_iters=8)
        print(">> mb, dr2")
        se_mb_.get_dr2_all(n_iters=8)

        print(">> mf, cvr2")
        se_mf_.get_cvr2_all(n_iters=8)
        print(">> mf, dr2")
        se_mf_.get_dr2_all(n_iters=8)

        print(">> full, cvr2")
        se_.get_cvr2_all(n_iters=8)
        print(">> full, dr2")
        se_.get_dr2_all(n_iters=8)
    except ValueError:
        print("not enough trials...")
        continue

    ses["full"].append(se_)
    ses["mb"].append(se_mb_)
    ses["mf"].append(se_mf_)

### cvr2

#### single session

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: se.cvr2_unit[regr][:, encoder.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.2, 0.8],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0)
            - se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.8, 0.8],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id / sess_id,
            fname=f"{regr}-cvr2_distro_full-{subj_id}_{sess_id}.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id / sess_id,
            fname=f"{regr}-cvr2_distro_strat-{subj_id}_{sess_id}.{fext}",
        )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff cvr2 between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

#### across sessions

In [ ]:
for regr in encoder.tv_keys:
    if regr != "block_side":
        regr_tex = regr.replace("_", r"\_")
        fig_full, ax_full = plot_kdes(
            data={
                reg: [
                    se_.cvr2_unit[regr][:, encoders["full"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    for i, se_ in enumerate(ses["full"])
                ]
                for reg in encoder.regions
            },
            do_sem=True,
            line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
            xlim=[-0.2, 0.8],
            label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
        )
        ax_full.axvline(x=0, color="#333333", linestyle="--")

        fig_strat, ax_strat = plot_kdes(
            data={
                reg: [
                    se_mb_.cvr2_unit[regr][:, encoders["mb"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    - se_mf_.cvr2_unit[regr][:, encoders["mf"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    for i, (se_mb_, se_mf_) in enumerate(zip(ses["mb"], ses["mf"]))
                ]
                for reg in encoder.regions
            },
            do_sem=True,
            line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
            xlim=[-0.8, 0.8],
            label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
        )
        ax_strat.axvline(x=0, color="#333333", linestyle="--")
        ax_strat.set_title(
            f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})"
        )

        for fext in ["svg", "png"]:
            save_fig(
                fig_full,
                FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id,
                fname=f"{regr}-cvr2_distro_full-{subj_id}_sessavg.{fext}",
            )
            save_fig(
                fig_strat,
                FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id,
                fname=f"{regr}-cvr2_distro_strat-{subj_id}_sessavg.{fext}",
            )

### delta r2

#### single session

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: se.dr2_unit[regr][:, encoder.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.05, 0.1],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0)
            - se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.3, 0.3],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id / sess_id,
            fname=f"{regr}-dr2_distro_full-{subj_id}_{sess_id}.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id / sess_id,
            fname=f"{regr}-dr2_distro_strat-{subj_id}_{sess_id}.{fext}",
        )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff dr2 between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

#### across sessions

In [ ]:
for regr in encoder.tv_keys:
    if regr != "block_side":
        regr_tex = regr.replace("_", r"\_")
        fig_full, ax_full = plot_kdes(
            data={
                reg: [
                    se_.dr2_unit[regr][:, encoders["full"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    for i, se_ in enumerate(ses["full"])
                ]
                for reg in encoder.regions
            },
            do_sem=True,
            line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
            xlim=[-0.05, 0.1],
            label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
        )
        ax_full.axvline(x=0, color="#333333", linestyle="--")

        fig_strat, ax_strat = plot_kdes(
            data={
                reg: [
                    se_mb_.dr2_unit[regr][:, encoders["mb"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    - se_mf_.dr2_unit[regr][:, encoders["mf"][i].reg_idxs[reg]].mean(
                        axis=0
                    )
                    for i, (se_mb_, se_mf_) in enumerate(zip(ses["mb"], ses["mf"]))
                ]
                for reg in encoder.regions
            },
            do_sem=True,
            line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
            xlim=[-0.3, 0.3],
            label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
        )
        ax_strat.axvline(x=0, color="#333333", linestyle="--")
        ax_strat.set_title(
            f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})"
        )

        for fext in ["svg", "png"]:
            save_fig(
                fig_full,
                FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id,
                fname=f"{regr}-dr2_distro_full-{subj_id}_sessavg.{fext}",
            )
            save_fig(
                fig_strat,
                FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id,
                fname=f"{regr}-dr2_distro_strat-{subj_id}_sessavg.{fext}",
            )

## sig pie chart

### single session

In [ ]:
from sg.models import BootstrapperShuffle as BSS

bss = BSS(
    subj_id,
    sess_id,
    Encoder,
    n=20,
    norm=True,
)
bss.get_ci_idxs()

bss_mb = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mb",
    n=20,
    norm=True,
)
bss_mb.get_ci_idxs()

bss_mf = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mf",
    n=20,
    norm=True,
)
bss_mf.get_ci_idxs()

In [ ]:
p_sig = {
    regr: {
        reg: len(bss.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

p_sig_mb = {
    regr: {
        reg: len(bss_mb.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

p_sig_mf = {
    regr: {
        reg: len(bss_mf.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

In [ ]:
import numpy as np

cids = {
    regr: {
        reg: np.sort(
            np.unique(
                np.union1d(bss_mb.ci_idxs_reg[regr][reg], bss_mf.ci_idxs_reg[regr][reg])
            )
        )
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

p_sig_union = {
    regr: {
        reg: len(cids[regr][reg]) / len(encoder.psths[reg]) for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

In [ ]:
from core.viz import plot_grouped_bar_h

fig_full, ax = plot_grouped_bar_h(
    data=p_sig, ylabel="p(significant) [full trial]", colors=colors_region, legend=False
)
ax.axvline(x=1.0, color="#333333", linewidth=0.75)

fig_union, ax = plot_grouped_bar_h(
    data=p_sig_union,
    ylabel="p(significant) [union]",
    colors=colors_region,
    legend=False,
)
ax.axvline(x=1.0, color="#333333", linewidth=0.75)

# for fext in ["svg", "png"]:
#     save_fig(
#         fig_full,
#         FIGURES_DIR / "bweight" / subj_id / sess_id / "p_significant",
#         f"p_significant_full-{subj_id}_{sess_id}.{fext}",
#     )
#     save_fig(
#         fig_union,
#         FIGURES_DIR / "bweight" / subj_id / sess_id / "p_significant",
#         f"p_significant_union-{subj_id}_{sess_id}.{fext}",
#     )

In [ ]:
# maybe because neurons in dms are more likely to encode task variables strongly in only one strategy, when you consider both strategies, those neurons become not significant

### all sessions

In [ ]:
from sg.models import BootstrapperShuffle as BSS

bsss = {k: [] for k in ["full", "mb", "mf"]}

p_sig_union_sess = {
    regr: {reg: [] for reg in encoder.regions} for regr in encoder.tv_keys
}

sess_idx = 0
for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(f">{sess_id}")
    try:
        bss_mb_ = BSS(
            subj_id,
            sess_id,
            StrategyEncoder,
            strategy_filter="mb",
            n=8,
            norm=True,
        )
        bss_mb_.get_ci_idxs()

        bss_mf_ = BSS(
            subj_id,
            sess_id,
            StrategyEncoder,
            strategy_filter="mf",
            n=8,
            norm=True,
        )
        bss_mf_.get_ci_idxs()

        bss_ = BSS(
            subj_id,
            sess_id,
            Encoder,
            n=8,
            norm=True,
        )
        bss_.get_ci_idxs()
    except ValueError:
        print("not enough trials")
        continue

    # union
    cids_ = {
        regr: {
            reg: np.sort(
                np.unique(
                    np.union1d(
                        bss_mb_.ci_idxs_reg[regr][reg], bss_mf_.ci_idxs_reg[regr][reg]
                    )
                )
            )
            for reg in encoder.regions
        }
        for regr in encoder.tv_keys
    }

    for regr in encoder.tv_keys:
        for reg in encoder.regions:
            p_sig_union_sess[regr][reg].append(
                len(cids_[regr][reg]) / len(encoders["full"][sess_idx].psths[reg])
            )

    bsss["full"].append(bss_)
    bsss["mb"].append(bss_mb_)
    bsss["mf"].append(bss_mf_)

    sess_idx += 1

In [ ]:
from core.viz import plot_grouped_bar_h
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

fig_union, ax = plot_grouped_bar_h(
    data=p_sig_union_sess,
    do_sem=True,
    ylabel="p(significant) [union]",
    colors=colors_region,
    legend=False,
)
ax.axvline(x=1.0, color="#333333", linewidth=0.75)

for fext in ["svg", "png"]:
    save_fig(
        fig_union,
        FIGURES_DIR / "bweight" / subj_id / "p_significant",
        f"p_significant_union-{subj_id}_sessavg.{fext}",
    )

# single neuron
filtered for significance in at least one strategy

## single session

In [ ]:
# save single session with bootstrapped error bars

from core.viz import plot_scatter

mn = min(np.min(encoder_mb.encoder_weights), np.min(encoder_mf.encoder_weights))
mx = max(np.max(encoder_mb.encoder_weights), np.max(encoder_mf.encoder_weights))

for regr in encoder.tv_keys:
    fig, axes = plt.subplots(
        ncols=2, figsize=(4.5, 2.5), sharey=True, tight_layout=True
    )

    for i, (ax, reg) in enumerate(zip(axes.flat, encoder.regions)):
        ax = plot_scatter(
            x=encoder_mb.encoder_weights[cids[regr][reg], encoder_mb.tv_idxs[regr]],
            y=encoder_mf.encoder_weights[cids[regr][reg], encoder_mf.tv_idxs[regr]],
            xerr=bss_mb.bweight_stats_emp["sem"][
                cids[regr][reg], encoder_mb.tv_idxs[regr]
            ],
            yerr=bss_mf.bweight_stats_emp["sem"][
                cids[regr][reg], encoder_mf.tv_idxs[regr]
            ],
            xlabel="mb",
            ylabel="mf",
            add_unity=True,
            add_lr=True,
            mn=mn,
            mx=mx,
            title=reg,
            ax=ax,
        )
    regr_tex = regr.replace("_", r"\_")
    fig.suptitle(rf"$\beta_{{\mathrm{{{regr_tex}}}}}$")

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "strategy_scatter" / subj_id / sess_id,
            f"{regr}-bweight_strategy_scatter-{subj_id}_{sess_id}.{fext}",
        )

## all sessions

### tavg

In [ ]:
# save scatter compiled across sessions, no errorbars (mess)
from core.data import subject_ids, session_ids

subj_id = "MR82"

bweights_master = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}

bweights_master_ns = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}

for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(sess_id)

    # get encoder weights
    encoder_ = Encoder(subj_id, sess_id, norm=True)
    encoder_mb_ = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
    encoder_mf_ = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

    try:
        encoder_mb_.fit_encoder()
        encoder_mf_.fit_encoder()
        encoder_.fit_encoder()
    except ValueError:
        continue

    # get cids
    bss_mb_ = BSS(
        subj_id,
        sess_id,
        StrategyEncoder,
        strategy_filter="mb",
        n=10,
        norm=True,
    )
    bss_mb_.get_ci_idxs()

    bss_mf_ = BSS(
        subj_id,
        sess_id,
        StrategyEncoder,
        strategy_filter="mf",
        n=10,
        norm=True,
    )
    bss_mf_.get_ci_idxs()

    cids_ = {
        regr: {
            reg: np.sort(
                np.unique(
                    np.union1d(
                        bss_mb_.ci_idxs_reg[regr][reg], bss_mf_.ci_idxs_reg[regr][reg]
                    )
                )
            )
            for reg in encoder_.regions
        }
        for regr in encoder_.tv_keys
    }

    cids_ns_ = {
        regr: {
            reg: np.sort(
                np.unique(np.setdiff1d(encoder_.reg_idxs[reg], cids_[regr][reg]))
            )
        }
        for regr in encoder_.tv_keys
    }

    # add to dict
    for regr in encoder_.tv_keys:
        for reg in encoder_.regions:
            bweights_master[regr][reg]["mb"].extend(
                encoder_mb_.encoder_weights[cids_[regr][reg], encoder_.tv_idxs[regr]]
            )
            bweights_master[regr][reg]["mf"].extend(
                encoder_mf_.encoder_weights[cids_[regr][reg], encoder_.tv_idxs[regr]]
            )

            bweights_master_ns[regr][reg]["mb"].extend(
                encoder_mb_.encoder_weights[cids_ns_[regr][reg], encoder_.tv_idxs[regr]]
            )
            bweights_master_ns[regr][reg]["mf"].extend(
                encoder_mf_.encoder_weights[cids_ns_[regr][reg], encoder_.tv_idxs[regr]]
            )

In [ ]:
bweights_master = {
    regr: {
        reg: {
            strat: np.array(bweights_master[regr][reg][strat]) for strat in ["mb", "mf"]
        }
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

In [ ]:
from core.viz import plot_scatter


def get_ssi(bweight_mb, bweight_mf, abs=False):
    if abs:
        return np.abs((bweight_mb - bweight_mf) / (bweight_mb + bweight_mf))
    else:
        return (bweight_mb - bweight_mf) / (bweight_mb + bweight_mf)


for regr in encoder.tv_keys:
    fig, axes = plt.subplots(
        ncols=2, figsize=(4.5, 2.5), sharey=True, tight_layout=True
    )

    for i, (ax, reg) in enumerate(zip(axes.flat, encoder.regions)):
        bw_mb = bweights_master[regr][reg]["mb"]
        bw_mf = bweights_master[regr][reg]["mf"]
        ax = plot_scatter(
            x=bw_mb,
            y=bw_mf,
            color=get_ssi(bw_mb, bw_mf, abs=True) >= 1,
            cmap="Set1_r",
            # color_log=True,
            # vmin=10**-2,
            # vmax=10**2,
            xlabel="mb",
            ylabel="mf",
            add_unity=True,
            add_lr=True,
            mn=mn,
            mx=mx,
            title=reg,
            ax=ax,
        )
    regr_tex = regr.replace("_", r"\_")
    fig.suptitle(rf"$\beta_{{\mathrm{{{regr_tex}}}}}$")

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "strategy_scatter" / subj_id,
            f"{regr}-bweight_strategy_scatter-highlighted-{subj_id}_sesscomp.{fext}",
        )

In [ ]:
from core.viz import plot_kdes
from core.utils import b_regr_tex

for regr in encoder.tv_keys:
    fig, ax = plot_kdes(
        data={
            reg: get_ssi(
                bweights_master[regr][reg]["mb"],
                bweights_master[regr][reg]["mf"],
                abs=False,
            )
            for reg in encoder.regions
        },
        bw_method=0.01,
        label="ssi",
        ylabel="density()",
        xlim=[-5, 5],
        ynorm=True,
        add_means=False,
        line_kwargs={reg: {"color": colors_region[reg]} for reg in encoder.regions},
    )
    ax.set_title(b_regr_tex(regr))

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "ssi" / "distros",
            fname=f"{regr}-bweight_ssi-{subj_id}_sesscomp.{fext}",
        )

In [ ]:
# check alpha similarity
# plot non significant cells on scatter

#### strategy interaction bweight distro

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=["response", "rewarded", "strategy", "response_prev", "rewarded_prev"],
    add_interaction=True,
    norm=True,
)
encoder.verify()

In [ ]:
strategy_xtion_bweights = {
    f"strategy_x_{regr}": {
        reg: encoder.encoder_weights[
            encoder.reg_idxs[reg],
            encoder.tv_idxs[encoder.get_xtion_key("strategy", regr)],
        ]
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
    if regr != "strategy"
}

In [ ]:
from core.viz import plot_grouped_violin

fig, ax = plot_grouped_violin(
    strategy_xtion_bweights, ylabel=r"$\beta$", colors=colors_region
)

for fext in ["svg", "png"]:
    save_fig(
        fig,
        FIGURES_DIR / "bweight" / "strategy_xtion" / subj_id / sess_id,
        fname=f"strategy_xtion_bweights-{subj_id}_{sess_id}.{fext}",
    )

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=Encoder,
    tv_keys=[
        "response",
        "rewarded",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
se.get_cvr2_all()
se.get_dr2_all()

In [ ]:
strategy_xtion_cvr2 = {
    f"strategy_x_{regr}": {
        reg: se.cvr2_unit[se.encoder_full.get_xtion_key("strategy", regr)][
            :, encoder.reg_idxs[reg]
        ].mean(axis=0)
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
    if regr != "strategy"
}

strategy_xtion_dr2 = {
    f"strategy_x_{regr}": {
        reg: se.dr2_unit[se.encoder_full.get_xtion_key("strategy", regr)][
            :, encoder.reg_idxs[reg]
        ].mean(axis=0)
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
    if regr != "strategy"
}

In [ ]:
fig_cvr2, ax = plot_grouped_violin(
    strategy_xtion_cvr2, ylabel=r"$\text{cv } r^2$", colors=colors_region
)
fig_dr2, ax = plot_grouped_violin(
    strategy_xtion_dr2, ylabel=r"$\Delta r^2$", colors=colors_region
)

for fext in ["svg", "png"]:
    save_fig(
        fig_cvr2,
        FIGURES_DIR / "cv_d_r2" / "strategy_xtion" / subj_id,
        f"strategy_xtion_cvr2-{subj_id}_{sess_id}.{fext}",
    )
    save_fig(
        fig_dr2,
        FIGURES_DIR / "cv_d_r2" / "strategy_xtion" / subj_id,
        f"strategy_xtion_dr2-{subj_id}_{sess_id}.{fext}",
    )

#### *attention!*: verify that axis neurons have larger beta weights for choice (correlate ssi and beta_strategy_interaction)

In [ ]:
from sg.models import StrategyEncoder

encoder_mb = StrategyEncoder(
    subj_id,
    sess_id,
    tv_keys=["response", "rewarded", "strategy", "response_prev", "rewarded_prev"],
    add_interaction=True,
    norm=True,
    strategy_filter="mb",
)
encoder_mf = StrategyEncoder(
    subj_id,
    sess_id,
    tv_keys=["response", "rewarded", "strategy", "response_prev", "rewarded_prev"],
    add_interaction=True,
    norm=True,
    strategy_filter="mf",
)

encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

In [ ]:
def get_ssi(bweight_mb, bweight_mf, abs=False):
    if abs:
        return np.abs((bweight_mb - bweight_mf) / (bweight_mb + bweight_mf))
    else:
        return (bweight_mb - bweight_mf) / (bweight_mb + bweight_mf)

In [ ]:
from sg.models import BootstrapperShuffle as BSS

bss_mb = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mb",
    n=8,
    norm=True,
)
bss_mb.get_ci_idxs()

bss_mf = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mf",
    n=8,
    norm=True,
)
bss_mf.get_ci_idxs()

In [ ]:
bss_mb.ci_idxs_reg["response"]

In [ ]:
cids = {
    regr: {
        reg: np.union1d(bss_mb.ci_idxs_reg[regr][reg], bss_mf.ci_idxs_reg[regr][reg])
        for reg in encoder.regions
    }
    for regr in ["response", "rewarded", "response_prev", "rewarded_prev"]
}

In [ ]:
# only get significant neurons

for regr in encoder.tv_keys:
    if regr == "strategy":
        continue
    fig, axes = plt.subplots(ncols=2, figsize=(5, 2), tight_layout=True)

    for i, reg in enumerate(encoder.regions):
        ssi = get_ssi(
            encoder_mb.encoder_weights[
                cids[regr][reg],
                encoder_mb.tv_idxs[encoder_mb.get_xtion_key("strategy", regr)],
            ],
            encoder_mf.encoder_weights[
                cids[regr][reg],
                encoder_mf.tv_idxs[encoder_mf.get_xtion_key("strategy", regr)],
            ],
            abs=True,
        )

        bweight_idxs = [
            idx
            for idx, reg_idx in enumerate(encoder.reg_idxs[reg])
            if reg_idx in cids[regr][reg]
        ]
        ax = plot_scatter(
            x=ssi,
            y=np.abs(strategy_xtion_bweights[f"strategy_x_{regr}"][reg][bweight_idxs]),
            xlabel="ssi",
            ylabel=f"strategy, {regr}",
            mn=-0.1,
            mx=2,
            add_lr=True,
            ax=axes[i],
        )
        ax.set_title(reg)

### tre

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

In [ ]:
from core.viz import plot_scatter

mn = min(np.min(encoder_mb.encoder_weights), np.min(encoder_mf.encoder_weights))
mx = max(np.max(encoder_mb.encoder_weights), np.max(encoder_mf.encoder_weights))

for regr in encoder.tv_keys:
    fig, axes = plt.subplots(
        ncols=2, figsize=(4.5, 2.5), sharey=True, tight_layout=True
    )

    for i, (ax, reg) in enumerate(zip(axes.flat, encoder.regions)):
        ax = plot_scatter(
            x=encoder_mb.encoder_weights[cids[regr][reg], encoder_mb.tv_idxs[regr]],
            y=encoder_mf.encoder_weights[cids[regr][reg], encoder_mf.tv_idxs[regr]],
            xerr=bss_mb.bweight_stats_emp["sem"][
                cids[regr][reg], encoder_mb.tv_idxs[regr]
            ],
            yerr=bss_mf.bweight_stats_emp["sem"][
                cids[regr][reg], encoder_mf.tv_idxs[regr]
            ],
            xlabel="mb",
            ylabel="mf",
            add_unity=True,
            add_lr=True,
            mn=mn,
            mx=mx,
            title=reg,
            ax=ax,
        )
    regr_tex = regr.replace("_", r"\_")
    fig.suptitle(rf"$\beta_{{\mathrm{{{regr_tex}}}}}$")

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "strategy_scatter" / subj_id / sess_id,
            f"{regr}-bweight_strategy_scatter-{subj_id}_{sess_id}.{fext}",
        )

## all sessions, jagged list
to check which cells are not significant, get the ssi sem across sessions, and check the alpha values between mb and mf

In [ ]:
# save scatter compiled across sessions, no errorbars (mess)
from core.data import subject_ids, session_ids
from sg.models import BootstrapperShuffle as BSS

subj_id = "MR82"

bweights_master = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}

bweights_master_ns = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}

alphas = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}

alphas_ns = {
    regr: {reg: {strat: [] for strat in ["mb", "mf"]} for reg in encoder.regions}
    for regr in encoder.tv_keys
}


for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(sess_id)

    # get encoder weights
    encoder_ = Encoder(subj_id, sess_id, norm=True)
    encoder_mb_ = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
    encoder_mf_ = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

    try:
        encoder_mb_.fit_encoder()
        encoder_mf_.fit_encoder()
        encoder_.fit_encoder()
    except ValueError:
        continue

    # get cids
    bss_mb_ = BSS(
        subj_id,
        sess_id,
        StrategyEncoder,
        strategy_filter="mb",
        n=10,
        norm=True,
    )
    bss_mb_.get_ci_idxs()

    bss_mf_ = BSS(
        subj_id,
        sess_id,
        StrategyEncoder,
        strategy_filter="mf",
        n=10,
        norm=True,
    )
    bss_mf_.get_ci_idxs()

    cids_ = {
        regr: {
            reg: np.sort(
                np.unique(
                    np.union1d(
                        bss_mb_.ci_idxs_reg[regr][reg], bss_mf_.ci_idxs_reg[regr][reg]
                    )
                )
            )
            for reg in encoder_.regions
        }
        for regr in encoder_.tv_keys
    }

    cids_ns_ = {
        regr: {
            reg: np.sort(
                np.unique(np.setdiff1d(encoder_.reg_idxs[reg], cids_[regr][reg]))
            )
            for reg in encoder_.regions
        }
        for regr in encoder_.tv_keys
    }

    # add to dict
    for regr in encoder_.tv_keys:
        for reg in encoder_.regions:
            bweights_master[regr][reg]["mb"].append(
                encoder_mb_.encoder_weights[cids_[regr][reg], encoder_.tv_idxs[regr]]
            )
            bweights_master[regr][reg]["mf"].append(
                encoder_mf_.encoder_weights[cids_[regr][reg], encoder_.tv_idxs[regr]]
            )
            alphas[regr][reg]["mb"].append(encoder_mb_.encoder.alpha_[cids_[regr][reg]])
            alphas[regr][reg]["mf"].append(encoder_mf_.encoder.alpha_[cids_[regr][reg]])

            bweights_master_ns[regr][reg]["mb"].append(
                encoder_mb_.encoder_weights[cids_ns_[regr][reg], encoder_.tv_idxs[regr]]
            )
            bweights_master_ns[regr][reg]["mf"].append(
                encoder_mf_.encoder_weights[cids_ns_[regr][reg], encoder_.tv_idxs[regr]]
            )
            alphas_ns[regr][reg]["mb"].append(
                encoder_mb_.encoder.alpha_[cids_ns_[regr][reg]]
            )
            alphas_ns[regr][reg]["mf"].append(
                encoder_mf_.encoder.alpha_[cids_ns_[regr][reg]]
            )

In [ ]:
mn = np.min(
    [
        np.min([np.min(a) for a in bweights_master[regr][reg][strat]])
        for regr in encoder.tv_keys
        for reg in encoder.regions
        for strat in ["mb", "mf"]
    ]
)
mx = np.max(
    [
        np.max([np.max(a) for a in bweights_master[regr][reg][strat]])
        for regr in encoder.tv_keys
        for reg in encoder.regions
        for strat in ["mb", "mf"]
    ]
)

In [ ]:
# plot significant and not significant in same scatter
from core.viz import plot_scatter
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for regr in encoder.tv_keys:
    fig, axes = plt.subplots(
        ncols=2, figsize=(4.5, 2.5), sharey=True, tight_layout=True
    )

    for i, (ax, reg) in enumerate(zip(axes.flat, encoder.regions)):
        # plot significant first
        bw_mb = np.concatenate(bweights_master[regr][reg]["mb"])
        bw_mf = np.concatenate(bweights_master[regr][reg]["mf"])
        ax = plot_scatter(
            x=bw_mb,
            y=bw_mf,
            color="#f0bb71",
            xlabel="mb",
            ylabel="mf",
            add_unity=True,
            add_lr=True,
            lr_color="#f26704",
            mn=mn,
            mx=mx,
            title=reg,
            ax=ax,
        )

        # then plot non significant
        bw_mb_ns = np.concatenate(bweights_master_ns[regr][reg]["mb"])
        bw_mf_ns = np.concatenate(bweights_master_ns[regr][reg]["mf"])
        ax = plot_scatter(
            x=bw_mb_ns,
            y=bw_mf_ns,
            color="#58606A",
            xlabel="mb",
            ylabel="mf",
            add_title=False,
            add_unity=True,
            add_lr=True,
            lr_color="#48556D",
            mn=mn,
            mx=mx,
            title=reg,
            ax=ax,
        )
    regr_tex = regr.replace("_", r"\_")
    fig.suptitle(rf"$\beta_{{\mathrm{{{regr_tex}}}}}$")

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "strategy_scatter" / subj_id,
            f"{regr}-bweight_strategy_scatter-highlighted-{subj_id}_sig_n_notsig_sesscomp.{fext}",
        )

In [ ]:
# ssi sem across sessions

from core.viz import plot_kdes
from core.utils import b_regr_tex
from utils.colors import colors_region, colors_strategy


def get_ssi(bweight_mb, bweight_mf, abs=False):
    if abs:
        return np.abs((bweight_mb - bweight_mf) / (bweight_mb + bweight_mf))
    else:
        return (bweight_mb - bweight_mf) / (bweight_mb + bweight_mf)


for regr in encoder.tv_keys:
    fig, ax = plot_kdes(
        data={
            reg: [
                get_ssi(
                    bweights_sess_mb,
                    bweights_sess_mf,
                    abs=False,
                )
                for (bweights_sess_mb, bweights_sess_mf) in zip(
                    bweights_master[regr][reg]["mb"], bweights_master[regr][reg]["mf"]
                )
            ]
            for reg in encoder.regions
        },
        do_sem=True,
        bw_method=0.1,
        label="ssi",
        ylabel="density",
        xlim=[-5, 5],
        ynorm=False,
        add_means=False,
        line_kwargs={reg: {"color": colors_region[reg]} for reg in encoder.regions},
    )
    ax.set_title(b_regr_tex(regr))
    ax.axvline(x=0, color="#555555", linewidth=0.75, linestyle="--", zorder=-1)
    ax.axvline(
        x=-1, color=colors_strategy["mf"], linewidth=0.75, linestyle="--", zorder=-1
    )
    ax.axvline(
        x=1, color=colors_strategy["mb"], linewidth=0.75, linestyle="--", zorder=-1
    )

    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "bweight" / "ssi" / "distros",
            fname=f"{regr}-bweight_ssi-{subj_id}_sessavg.{fext}",
        )

In [ ]:
# alpha values between mb and mf (significant cells only)
from core.utils import b_regr_tex
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

reg_exp = encoder.max_reg
alpha_range = np.logspace(-reg_exp, reg_exp, 2 * reg_exp + 1)

for regr in encoder.tv_keys:
    fig, axes = plt.subplots(ncols=2, figsize=(5, 2), tight_layout=True)
    for i, reg in enumerate(encoder.regions):
        ax = axes[i]

        a_mb = np.concatenate(alphas[regr][reg]["mb"])
        a_mf = np.concatenate(alphas[regr][reg]["mf"])

        alpha_counts = np.array(
            [
                [len(np.where((a_mb == a1) & (a_mf == a2))[0]) for a2 in alpha_range]
                for a1 in alpha_range
            ]
        )

        im = ax.imshow(alpha_counts, cmap="Blues")

        ax.set_xlabel(r"mb $\alpha$")
        ax.set_ylabel(r"mf $\alpha$")
        ax.set_title(rf"{reg}, $\alpha$ count comparison", fontsize=7)

        ax.set_xticks(
            np.arange(len(alpha_range)),
            [f"{a:.0e}" for a in alpha_range],
            rotation=45,
            ha="right",
            fontsize=5,
        )
        ax.set_yticks(
            np.arange(len(alpha_range)), [f"{a:.0e}" for a in alpha_range], fontsize=5
        )

        fig.colorbar(im, ax=ax, shrink=0.67, label="count")
    fig.suptitle(rf"{b_regr_tex(regr)}, significant units")
    for fext in ["svg", "png"]:
        save_fig(
            fig,
            FIGURES_DIR / "alphas" / subj_id,
            fname=f"{regr}-alphas_strategy_comp-{subj_id}_sesscomp.{fext}",
        )

In [ ]:
# verify that axes ssi have large strategy interaction beta weights
# > just do for one session to start
# correlate ssi with strategy interaction beta weight
